# Later BI claims extract: aggregate validation and emergence decomposition

This notebook checks the later BI claims extract against the original dataset at the common
2022 Q4 valuation date, then measures what happened between 2022 Q4 and the later observed
position. It works at aggregate level in the public release: the record-level repair logic and
individual claim identifiers are intentionally excluded.

Some internal filenames still contain `V11` because that was the working label for the later
extract; the dissertation refers to it as the later BI claims extract.

> **Public release note.** Notebook outputs and execution counts have been removed because the underlying Malaysian Motor claims data are confidential. Local user-specific paths and record identifiers have also been removed. The code documents the analysis workflow but cannot be executed end-to-end without appropriately structured confidential input data.


## 0. Set up the data locations

This section loads the required packages and defines the known locations of the original
master claims history and the validated later-extract Parquet file. It checks that both
confidential inputs are present before any reconciliation is attempted.

In [ ]:
from pathlib import Path
import gc
import polars as pl
import pandas as pd

PROJECT_DIR = Path("/path/to/BI_large_claims_project")
PROCESSED_DIR = PROJECT_DIR / "processed"

OLD_MASTER_PARQUET = PROJECT_DIR / "BI_large_claims_master_long.parquet"
REPAIRED_PARQUET = PROCESSED_DIR / "BI_LARGE_CLAIMS_IFOA_REPAIRED.parquet"

for label, path in {
    "OLD_MASTER_PARQUET": OLD_MASTER_PARQUET,
    "REPAIRED_PARQUET": REPAIRED_PARQUET,
}.items():
    print(f"{label:<24} exists={path.exists()}  {path}")

if not OLD_MASTER_PARQUET.exists() or not REPAIRED_PARQUET.exists():
    raise FileNotFoundError(
        "Set PROJECT_DIR to the confidential project root and ensure both "
        "the original master and validated later-extract Parquet files exist."
    )


## 1. Compare latest financial positions by accident year

For each accident year, the code takes the latest available development record for every
claim and totals incurred BI, BI Capped, BI Excess and paid amounts. The same calculation
is run on the original and later extracts so their latest portfolio positions can be compared
on a consistent basis.

In [ ]:
def latest_financial_summary_by_year(
    parquet_path,
    year,
    old_version=False,
):
    """
    Read one accident year, select the latest DEV_QTR for each
    within-version claim identity, and return financial totals.
    """

    lf = pl.scan_parquet(parquet_path)

    if old_version:
        lf = lf.rename({"cover": "COVER"})

    claim_id = [
        "CLASS",
        "COVER",
        "NATLOSS",
        "ACC_YEAR",
        "ACC_QTR",
        "CLAIMS_KEY",
    ]

    df = (
        lf
        .filter(pl.col("ACC_YEAR") == year)
        .select(
            claim_id
            + [
                "DEV_QTR",
                "CUM_INC_AMT",
                "CUM_INC_NON_LARGE",
                "CUM_INC_LARGE",
                "CUM_PAIDLS",
            ]
        )
        .collect()
    )

    latest = (
        df
        .sort(claim_id + ["DEV_QTR"])
        .group_by(claim_id, maintain_order=True)
        .last()
    )

    result = latest.select(
        [
            pl.len().alias("latest_records"),

            pl.col("CUM_INC_AMT")
            .fill_null(0)
            .sum()
            .alias("cum_inc_amt"),

            pl.col("CUM_INC_NON_LARGE")
            .fill_null(0)
            .sum()
            .alias("bic"),

            pl.col("CUM_INC_LARGE")
            .fill_null(0)
            .sum()
            .alias("bixs"),

            pl.col("CUM_PAIDLS")
            .fill_null(0)
            .sum()
            .alias("paid"),

            (
                pl.col("CUM_INC_LARGE").fill_null(0) > 0
            )
            .sum()
            .alias("bixs_records"),

            pl.col("DEV_QTR")
            .max()
            .alias("max_dev_qtr"),
        ]
    )

    out = result.to_dicts()[0]
    out["ACC_YEAR"] = year

    del df, latest
    gc.collect()

    return out

In [ ]:
old_results = []
new_results = []

for year in range(2010, 2027):

    print(f"\nAccident year {year}")

    if year <= 2022:
        print("  OLD...")
        old_results.append(
            latest_financial_summary_by_year(
                OLD_MASTER_PARQUET,
                year,
                old_version=True,
            )
        )

    print("  NEW...")
    new_results.append(
        latest_financial_summary_by_year(
            REPAIRED_PARQUET,
            year,
            old_version=False,
        )
    )

print("\nFinished.")

In [ ]:
old_results_df = pl.DataFrame(old_results)
new_results_df = pl.DataFrame(new_results)

financial_compare = (
    old_results_df
    .join(
        new_results_df,
        on="ACC_YEAR",
        how="full",
        suffix="_new",
        coalesce=True,
    )
    .sort("ACC_YEAR")
)

display(financial_compare)

## 2. Reconstruct the original 2022 Q4 valuation diagonal

The original dataset is used to recover the exact development quarter available for every
accident-year/accident-quarter cell at 2022 Q4. Those maturity points are then applied to
both datasets, ensuring that the comparison at the valuation date is made at exactly the
same development ages.

In [ ]:
old_diagonal = (
    pl.scan_parquet(OLD_MASTER_PARQUET)
    .group_by(["ACC_YEAR", "ACC_QTR"])
    .agg(
        pl.col("DEV_QTR").max().alias("OLD_MAX_DEV_QTR")
    )
    .sort(["ACC_YEAR", "ACC_QTR"])
    .collect()
)

display(old_diagonal)

In [ ]:
old_diagonal_map = {
    (row["ACC_YEAR"], row["ACC_QTR"]): row["OLD_MAX_DEV_QTR"]
    for row in old_diagonal.to_dicts()
}

print("Number of valuation cells:", len(old_diagonal_map))

print("\nFirst few:")
for key in list(old_diagonal_map.keys())[:8]:
    print(key, "-> DEV_QTR", old_diagonal_map[key])

print("\nLast few:")
for key in list(old_diagonal_map.keys())[-8:]:
    print(key, "-> DEV_QTR", old_diagonal_map[key])

In [ ]:
import gc

def valuation_diagonal_summary_by_year(
    parquet_path,
    year,
    diagonal_map,
    old_version=False,
):
    """
    Financial position for one accident year at the historical
    2022 Q4 valuation diagonal.

    Each accident quarter is filtered to its exact DEV_QTR
    on the valuation diagonal.
    """

    lf = pl.scan_parquet(parquet_path)

    if old_version:
        lf = lf.rename({"cover": "COVER"})

    quarter_frames = []

    for qtr in [1, 2, 3, 4]:

        target_dev = diagonal_map.get((year, qtr))

        if target_dev is None:
            continue

        q = (
            lf
            .filter(
                (pl.col("ACC_YEAR") == year)
                & (pl.col("ACC_QTR") == qtr)
                & (pl.col("DEV_QTR") == target_dev)
            )
            .select(
                [
                    "CLASS",
                    "COVER",
                    "NATLOSS",
                    "ACC_YEAR",
                    "ACC_QTR",
                    "CLAIMS_KEY",
                    "DEV_QTR",
                    "CUM_INC_AMT",
                    "CUM_INC_NON_LARGE",
                    "CUM_INC_LARGE",
                    "CUM_PAIDLS",
                ]
            )
            .collect()
        )

        quarter_frames.append(q)

    if not quarter_frames:
        return None

    diagonal = pl.concat(quarter_frames)

    result = diagonal.select(
        [
            pl.len().alias("records"),

            pl.col("CUM_INC_AMT")
            .fill_null(0)
            .sum()
            .alias("cum_inc_amt"),

            pl.col("CUM_INC_NON_LARGE")
            .fill_null(0)
            .sum()
            .alias("bic"),

            pl.col("CUM_INC_LARGE")
            .fill_null(0)
            .sum()
            .alias("bixs"),

            pl.col("CUM_PAIDLS")
            .fill_null(0)
            .sum()
            .alias("paid"),

            (
                pl.col("CUM_INC_LARGE")
                .fill_null(0) > 0
            )
            .sum()
            .alias("bixs_records"),
        ]
    )

    out = result.to_dicts()[0]
    out["ACC_YEAR"] = year

    del diagonal, quarter_frames
    gc.collect()

    return out

In [ ]:
old_diag_results = []
v11_diag_results = []

for year in range(2010, 2023):

    print(f"AY {year}")

    old_diag_results.append(
        valuation_diagonal_summary_by_year(
            OLD_MASTER_PARQUET,
            year,
            old_diagonal_map,
            old_version=True,
        )
    )

    v11_diag_results.append(
        valuation_diagonal_summary_by_year(
            REPAIRED_PARQUET,
            year,
            old_diagonal_map,
            old_version=False,
        )
    )

print("Finished.")

## 3. Reconcile the 2022 Q4 starting position and later emergence

This section compares the original 2022 Q4 BI Excess position with the same diagonal
reconstructed from the later extract. The difference is treated as a restatement of the
starting position, while subsequent movement from the restated 2022 Q4 amount to the
later observed amount is treated as genuine post-valuation emergence.

In [ ]:
old_diag_df = pl.DataFrame(old_diag_results)
v11_diag_df = pl.DataFrame(v11_diag_results)

v11_latest_2010_2022 = (
    new_results_df
    .filter(pl.col("ACC_YEAR") <= 2022)
)

validation_bridge = (
    old_diag_df
    .select(
        [
            "ACC_YEAR",
            pl.col("records").alias("old_records_2022q4"),
            pl.col("bixs").alias("old_bixs_2022q4"),
            pl.col("bic").alias("old_bic_2022q4"),
            pl.col("cum_inc_amt").alias("old_bi_2022q4"),
        ]
    )
    .join(
        v11_diag_df.select(
            [
                "ACC_YEAR",
                pl.col("records").alias("v11_records_2022q4"),
                pl.col("bixs").alias("v11_bixs_2022q4"),
                pl.col("bic").alias("v11_bic_2022q4"),
                pl.col("cum_inc_amt").alias("v11_bi_2022q4"),
            ]
        ),
        on="ACC_YEAR",
    )
    .join(
        v11_latest_2010_2022.select(
            [
                "ACC_YEAR",
                pl.col("latest_records").alias("v11_latest_records"),
                pl.col("bixs").alias("v11_latest_bixs"),
                pl.col("bic").alias("v11_latest_bic"),
                pl.col("cum_inc_amt").alias("v11_latest_bi"),
            ]
        ),
        on="ACC_YEAR",
    )
    .with_columns(
        [
            (
                pl.col("v11_bixs_2022q4")
                - pl.col("old_bixs_2022q4")
            ).alias("bixs_restatement"),

            (
                pl.col("v11_latest_bixs")
                - pl.col("v11_bixs_2022q4")
            ).alias("bixs_post_2022_emergence"),

            (
                pl.col("v11_latest_bixs")
                - pl.col("old_bixs_2022q4")
            ).alias("bixs_total_old_to_latest"),
        ]
    )
    .sort("ACC_YEAR")
)

display(validation_bridge)

In [ ]:
validation_summary = (
    validation_bridge
    .with_columns(
        [
            (
                100
                * pl.col("bixs_restatement")
                / pl.col("old_bixs_2022q4")
            ).alias("bixs_restatement_pct"),

            (
                100
                * pl.col("bixs_post_2022_emergence")
                / pl.col("v11_bixs_2022q4")
            ).alias("bixs_emergence_pct"),

            (
                pl.col("v11_latest_bixs")
                / pl.col("v11_bixs_2022q4")
            ).alias("bixs_development_ratio"),
        ]
    )
    .select(
        [
            "ACC_YEAR",
            "old_bixs_2022q4",
            "v11_bixs_2022q4",
            "v11_latest_bixs",
            "bixs_restatement",
            "bixs_restatement_pct",
            "bixs_post_2022_emergence",
            "bixs_emergence_pct",
            "bixs_development_ratio",
        ]
    )
    .sort("ACC_YEAR")
)

display(validation_summary)

In [ ]:
portfolio_bridge = validation_summary.select(
    [
        pl.col("old_bixs_2022q4").sum().alias("old_bixs_2022q4"),
        pl.col("v11_bixs_2022q4").sum().alias("v11_bixs_2022q4"),
        pl.col("v11_latest_bixs").sum().alias("v11_latest_bixs"),
        pl.col("bixs_restatement").sum().alias("total_restatement"),
        pl.col("bixs_post_2022_emergence").sum().alias("total_post_2022_emergence"),
    ]
)

display(portfolio_bridge)

In [ ]:
output_path = (
    PROCESSED_DIR
    / "V11_2022Q4_to_latest_BIXS_validation_bridge.csv"
)

validation_summary.write_csv(output_path)

print("Saved:", output_path)

### Check the historical valuation-diagonal files

The saved one-year and two-year historical prediction files are checked to confirm that
they contain a single valuation diagonal with the expected accident-year and development
structure. This is an audit check only; it does not change the later-extract results.

In [ ]:
PRED_1 = (
    PROCESSED_DIR
    / "chapter5_outputs"
    / "section_5_9A_historical_valuation_diagonal"
    / "historical_diagonal_claim_level_predictions.parquet"
)

PRED_2 = (
    PROCESSED_DIR
    / "chapter5_outputs"
    / "section_5_9C_historical_valuation_diagonal_2yr"
    / "historical_diagonal_claim_level_predictions.parquet"
)

for name, path in {
    "Historical diagonal": PRED_1,
    "Historical diagonal 2yr": PRED_2,
}.items():
    print("\n", name)
    print("Path:", path)
    print("Exists:", path.exists())

    if path.exists():
        schema = pl.read_parquet_schema(path)
        print("Columns:")
        for col, dtype in schema.items():
            print(f"  {col:<40} {dtype}")

In [ ]:
for name, path in {
    "1yr": PRED_1,
    "2yr": PRED_2,
}.items():

    print(f"\n--- {name} historical diagonal ---")

    diag_check = (
        pl.scan_parquet(path)
        .filter(pl.col("included_in_main_diagonal") == True)
        .select(
            [
                pl.col("VALUATION_QTR_INDEX").n_unique()
                    .alias("n_valuation_indices"),
                pl.col("VALUATION_QTR_INDEX").min()
                    .alias("min_valuation_index"),
                pl.col("VALUATION_QTR_INDEX").max()
                    .alias("max_valuation_index"),
                pl.col("ACC_YEAR").min().alias("min_acc_year"),
                pl.col("ACC_YEAR").max().alias("max_acc_year"),
                pl.len().alias("rows"),
            ]
        )
        .collect()
    )

    display(diag_check)

    # Show the valuation diagonal itself
    diag_pattern = (
        pl.scan_parquet(path)
        .filter(pl.col("included_in_main_diagonal") == True)
        .group_by(["ACC_YEAR", "ACC_QTR"])
        .agg(
            pl.col("DEV_QTR").unique().sort().alias("DEV_QTR")
        )
        .sort(["ACC_YEAR", "ACC_QTR"])
        .collect()
    )

    display(diag_pattern.head(8))
    display(diag_pattern.tail(8))

## 4. Decompose later BI Excess emergence

The later BI Excess movement is split into three actuarially different sources: development
on claims already in BI Excess at 2022 Q4, reported BI Capped claims that later cross the
MYR 500,000 threshold, and claims not present at 2022 Q4 that later emerge as BI Excess
(an operational pure IBNR proxy). The decomposition is calculated by accident year and
then aggregated to portfolio level.

In [ ]:
def v11_bixs_emergence_by_year(
    parquet_path,
    year,
    diagonal_map,
):
    """
    Decompose later-extract BIXS development from the reconstructed
    2022 Q4 valuation position to the latest later-extract observation.

    Components:
      1. Claims already BIXS at 2022 Q4
      2. Reported non-BIXS claims crossing to BIXS
      3. Claims absent at 2022 Q4 subsequently appearing as BIXS
    """

    lf = pl.scan_parquet(parquet_path)

    claim_id = [
        "CLASS",
        "COVER",
        "NATLOSS",
        "ACC_YEAR",
        "ACC_QTR",
        "CLAIMS_KEY",
    ]

    # Bring only one accident year into memory.
    df = (
        lf
        .filter(pl.col("ACC_YEAR") == year)
        .select(
            claim_id
            + [
                "DEV_QTR",
                "CUM_INC_LARGE",
            ]
        )
        .collect()
    )

    # Latest position for every claim.
    latest = (
        df
        .sort(claim_id + ["DEV_QTR"])
        .group_by(claim_id, maintain_order=True)
        .last()
        .select(
            claim_id
            + [
                pl.col("CUM_INC_LARGE")
                .fill_null(0)
                .alias("LATEST_BIXS")
            ]
        )
    )

    # Reconstruct exact 2022 Q4 diagonal.
    t0_parts = []

    for qtr in [1, 2, 3, 4]:

        target_dev = diagonal_map.get((year, qtr))

        if target_dev is None:
            continue

        part = (
            df
            .filter(
                (pl.col("ACC_QTR") == qtr)
                & (pl.col("DEV_QTR") == target_dev)
            )
            .select(
                claim_id
                + [
                    pl.col("CUM_INC_LARGE")
                    .fill_null(0)
                    .alias("T0_BIXS")
                ]
            )
            .with_columns(
                pl.lit(True).alias("PRESENT_AT_T0")
            )
        )

        t0_parts.append(part)

    t0 = pl.concat(t0_parts)

    # Every T0 claim must also occur in latest;
    # claims with no T0 match are post-valuation reports.
    bridge = (
        latest
        .join(
            t0,
            on=claim_id,
            how="left",
        )
        .with_columns(
            [
                pl.col("T0_BIXS")
                .fill_null(0),

                pl.col("PRESENT_AT_T0")
                .fill_null(False),
            ]
        )
        .with_columns(
            [
                pl.when(
                    pl.col("T0_BIXS") > 0
                )
                .then(pl.lit("BIXS_AT_2022Q4"))

                .when(
                    pl.col("PRESENT_AT_T0")
                    & (pl.col("LATEST_BIXS") > 0)
                )
                .then(pl.lit("REPORTED_BIC_TO_BIXS"))

                .when(
                    (~pl.col("PRESENT_AT_T0"))
                    & (pl.col("LATEST_BIXS") > 0)
                )
                .then(pl.lit("PURE_IBNR_PROXY"))

                .otherwise(pl.lit("NO_BIXS_EMERGENCE"))
                .alias("EMERGENCE_TYPE")
            ]
        )
        .with_columns(
            (
                pl.col("LATEST_BIXS")
                - pl.col("T0_BIXS")
            ).alias("BIXS_DEVELOPMENT")
        )
    )

    summary = (
        bridge
        .group_by("EMERGENCE_TYPE")
        .agg(
            [
                pl.len().alias("claim_count"),

                pl.col("T0_BIXS")
                .sum()
                .alias("t0_bixs"),

                pl.col("LATEST_BIXS")
                .sum()
                .alias("latest_bixs"),

                pl.col("BIXS_DEVELOPMENT")
                .sum()
                .alias("bixs_development"),
            ]
        )
        .with_columns(
            pl.lit(year).alias("ACC_YEAR")
        )
    )

    del df, latest, t0, bridge
    gc.collect()

    return summary

In [ ]:
all_emergence_results = []

for year in range(2010, 2023):
    print(f"Processing AY {year}...")

    result = v11_bixs_emergence_by_year(
        REPAIRED_PARQUET,
        year,
        old_diagonal_map,
    )

    all_emergence_results.append(result)

emergence_all = (
    pl.concat(all_emergence_results)
    .sort(["ACC_YEAR", "EMERGENCE_TYPE"])
)

display(emergence_all)

print("Finished.")

In [ ]:
emergence_by_ay = (
    emergence_all
    .pivot(
        values="bixs_development",
        index="ACC_YEAR",
        on="EMERGENCE_TYPE",
        aggregate_function="sum",
    )
    .fill_null(0)
    .sort("ACC_YEAR")
)

display(emergence_by_ay)

In [ ]:
component_cols = [
    c for c in [
        "BIXS_AT_2022Q4",
        "REPORTED_BIC_TO_BIXS",
        "PURE_IBNR_PROXY",
        "NO_BIXS_EMERGENCE",
    ]
    if c in emergence_by_ay.columns
]

emergence_by_ay = (
    emergence_by_ay
    .with_columns(
        pl.sum_horizontal(component_cols)
        .alias("TOTAL_POST_2022_BIXS_EMERGENCE")
    )
)

display(emergence_by_ay)

In [ ]:
portfolio_emergence = (
    emergence_all
    .group_by("EMERGENCE_TYPE")
    .agg(
        [
            pl.col("claim_count").sum().alias("claim_count"),
            pl.col("t0_bixs").sum().alias("t0_bixs"),
            pl.col("latest_bixs").sum().alias("latest_bixs"),
            pl.col("bixs_development").sum().alias("bixs_development"),
        ]
    )
    .sort("EMERGENCE_TYPE")
)

display(portfolio_emergence)

print(
    "Total post-2022 BIXS emergence:",
    portfolio_emergence["bixs_development"].sum()
)

### Save the emergence results

The accident-year and portfolio decomposition tables are saved for use in the later
out-of-time validation notebook and in the dissertation results.

In [ ]:
emergence_by_ay.write_csv(
    PROCESSED_DIR
    / "V11_2022Q4_to_latest_BIXS_emergence_decomposition_by_AY.csv"
)

portfolio_emergence.write_csv(
    PROCESSED_DIR
    / "V11_2022Q4_to_latest_BIXS_emergence_decomposition_portfolio.csv"
)

print("Saved decomposition outputs.")